# YOLO 車牌辨識 Python 語法互動式逐區塊實作

本 Notebook 與 Word 教學手冊同步。每個單元依序包含：

1. 執行前預測
2. 可執行程式
3. 逐行語法解說
4. 修改挑戰
5. 驗收區塊

> 建議：不要使用「全部執行」。先寫下預測，再逐格執行。模型訓練預設關閉。


## 0 課程導航與環境

資料流：完整影像 → YOLO11 車牌框 → crop → YOLOv8 字元框 → 排序 → 車牌字串 → UI 事件 → 評估

以下安裝格在 Colab 執行；一般 Python 語法單元不需要 GPU。


In [ ]:
# Colab 套件安裝：第一次執行約需數分鐘
%pip install -q "ultralytics>=8.3.0" easyocr huggingface_hub opencv-python-headless pyyaml matplotlib pandas


In [ ]:
# 環境檢查
import sys, platform, re, urllib.request
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt, torch
print("Python:", sys.version)
print("Platform:", platform.platform())
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("推論裝置:", DEVICE)


# 第 0 單元：先完成一次真正的車牌辨識

本單元先建立成功經驗，再於第 1～13 單元逐步拆解 Python 語法。

資料流：公開 YOLOv8 車牌模型 → 車牌框 → ROI 裁切 → EasyOCR → 正規化字串 → 畫框與紀錄。

> 公開模型是課堂基準，不代表適合所有國家、角度與夜間場景。台灣車牌專題請使用後面的自訂訓練區塊。


## 0-1 下載公開車牌權重

執行前預測：`hf_hub_download()` 回傳的是模型物件、網址，還是本機檔案路徑？


In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

PUBLIC_REPO = "Koushim/yolov8-license-plate-detection"
PUBLIC_FILE = "best.pt"
public_weights = hf_hub_download(repo_id=PUBLIC_REPO, filename=PUBLIC_FILE)
plate_model = YOLO(public_weights)
print("權重路徑:", public_weights)
print("模型類別:", plate_model.names)


### 逐行重點

- `PUBLIC_REPO` 與 `PUBLIC_FILE` 是集中管理的字串常數。
- `hf_hub_download()` 下載檔案並回傳 Colab 本機快取路徑。
- `YOLO(public_weights)` 建立模型物件；真正推論要呼叫 `.predict()`。
- `plate_model.names` 用來驗證模型類別，不應只相信檔名。


## 0-2 自動下載公開測試影像

程式會依序嘗試兩個公開範例；若都失敗，下一格可上傳自己的照片。


In [ ]:
DEMO_URLS = [
    "https://raw.githubusercontent.com/yihong1120/YOLOv8-License-Plate-Insights/main/medias/Scooter.jpg",
    "https://raw.githubusercontent.com/DefKd911/License-Plate-Detection-and-Recognition-Yolov8/main/demo_frame.jpg",
]
DEMO_IMAGE = Path("/content/demo_plate.jpg")

downloaded = False
for url in DEMO_URLS:
    try:
        urllib.request.urlretrieve(url, DEMO_IMAGE)
        image_bgr = cv2.imread(str(DEMO_IMAGE))
        if image_bgr is not None:
            downloaded = True
            print("測試影像:", url, image_bgr.shape)
            break
    except Exception as error:
        print("此來源暫時無法下載:", type(error).__name__)

if downloaded:
    plt.figure(figsize=(11, 7))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
else:
    print("請執行下一個上傳區塊。")


## 0-3 備援：上傳自己的車牌照片

只有上一格兩個網址都失敗，或想測試自己的照片時才需要執行。請避免公開含個人資料的真實車牌影像。


In [ ]:
from google.colab import files

uploaded = files.upload()
if uploaded:
    filename, payload = next(iter(uploaded.items()))
    image_bgr = cv2.imdecode(np.frombuffer(payload, np.uint8), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"無法解碼影像: {filename}")
    print(filename, image_bgr.shape)


## 0-4 初始化 OCR 並定義完整 ALPR 函式

第一次建立 `Reader` 時會下載 OCR 權重。`YOLO` 解決定位，`EasyOCR` 解決文字辨識。


In [ ]:
import easyocr

reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

def normalize_plate(text):
    """轉大寫並移除非英數字元。"""
    return re.sub(r"[^A-Z0-9]", "", text.upper())

def run_alpr(image_bgr, plate_model, reader, conf=0.25):
    """輸入 BGR 影像；回傳畫框影像與辨識紀錄。"""
    result = plate_model.predict(
        image_bgr, conf=conf, device=DEVICE, verbose=False
    )[0]
    annotated = image_bgr.copy()
    records = []
    if result.boxes is None or len(result.boxes) == 0:
        return annotated, records

    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    height, width = image_bgr.shape[:2]

    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = map(int, box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(width, x2), min(height, y2)
        crop = image_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        ocr_items = reader.readtext(
            crop,
            detail=1,
            paragraph=False,
            allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-",
        )
        raw_text = max(ocr_items, key=lambda item: item[2])[1] if ocr_items else ""
        ocr_conf = max((float(item[2]) for item in ocr_items), default=0.0)
        plate_text = normalize_plate(raw_text)

        records.append({
            "plate": plate_text,
            "raw_text": raw_text,
            "plate_conf": float(score),
            "ocr_conf": ocr_conf,
            "box": (x1, y1, x2, y2),
        })
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 3)
        label = plate_text or f"plate {score:.2f}"
        cv2.putText(annotated, label, (x1, max(25, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    return annotated, records


## 0-5 執行推論並顯示結果


In [ ]:
if "image_bgr" not in globals() or image_bgr is None:
    raise RuntimeError("尚未取得測試影像，請先執行 0-2 或 0-3。")

annotated, records = run_alpr(image_bgr, plate_model, reader, conf=0.25)
print("辨識紀錄:", records)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

assert isinstance(records, list)
print("驗收完成：程式已回傳 records list；若為空，請換圖或把 conf 降至 0.15。")


## 0-6 自訂資料集訓練與模型切換

資料夾應包含 `images/train`、`images/val`、`labels/train`、`labels/val` 與 `data.yaml`。第一次上課維持 `RUN_TRAINING=False`；完成資料準備後才切換為 `True`。


In [ ]:
RUN_TRAINING = False
CUSTOM_YAML = Path("/content/plate_dataset/data.yaml")

if RUN_TRAINING:
    if not CUSTOM_YAML.exists():
        raise FileNotFoundError(f"找不到 {CUSTOM_YAML}")
    base_model = YOLO("yolo11n.pt")
    train_result = base_model.train(
        data=str(CUSTOM_YAML),
        epochs=100,
        imgsz=960,
        batch=8,
        device=DEVICE,
        project="/content/runs",
        name="taiwan_plate_yolo11n",
        patience=20,
        seed=42,
    )
    custom_weights = Path(train_result.save_dir) / "weights" / "best.pt"
    assert custom_weights.exists()
    plate_model = YOLO(str(custom_weights))
    print("已切換自訓模型:", custom_weights)
else:
    print("目前使用公開基準模型；資料集完成後再開啟訓練。")


# 第 1 單元 專案資料流與 Python 資料型態

**學習目標**：用變數、型態、list、tuple 與 dict 描述車牌辨識系統中的資料。

**語法焦點**：變數賦值、type、float、bool、str、tuple、list、dict、f string

## 執行前預測

- [ ] plate_conf、two_lines、box 的型態各是什麼？
- [ ] 第二個 print 的完整輸出是什麼？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
plate_conf = 0.50
char_conf = 0.35
two_lines = False
model_name = "yolo11n.pt"
box = (402, 287, 615, 356)
characters = ["A", "B", "C", "1", "2", "3", "4"]
record = {"plate": "ABC1234", "accepted": True}

print(type(plate_conf), type(two_lines), type(box))
print(f"{record['plate']} DET {plate_conf:.2f}")


## 逐行語法解說

- **第 1 行 `plate_conf = 0.50`**：以 = 將 float 儲存到變數；它是車牌框門檻。
- **第 2 行 `char_conf = 0.35`**：字元門檻獨立管理，避免與車牌門檻混淆。
- **第 3 行 `False`**：布林值不是字串，首字母必須大寫。
- **第 4 行 `字串`**：模型檔名是 str。
- **第 5 行 `tuple`**：四個固定座標組成不可直接修改的 tuple。
- **第 6 行 `list`**：字元數量與順序可變，因此使用 list。
- **第 7 行 `dict`**：用具名 key 描述一筆辨識紀錄。
- **第 9 行 `type`**：檢查執行時資料型態。
- **第 10 行 `f string`**：大括號嵌入值，:.2f 顯示兩位小數。


## 修改挑戰

- [ ] 替 record 新增 plate_conf 欄位。
- [ ] 把 box 改為 list，修改第一個座標並比較 tuple。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert record["plate"] == "ABC1234"


# 第 2 單元 路徑 裝置與條件運算式

**學習目標**：集中管理專案路徑、GPU 裝置與信心門檻。

**語法焦點**：import、Path、/ 運算子、條件運算式、mkdir、常數命名

## 執行前預測

- [ ] gpu_available=False 時 DEVICE 為何？
- [ ] 重跑 mkdir 是否會錯誤？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
from pathlib import Path

PROJECT = Path("/content/plate_project")
MODELS = PROJECT / "models"
gpu_available = False  # Colab 正式版改用 torch.cuda.is_available()
DEVICE = 0 if gpu_available else "cpu"
MODELS.mkdir(parents=True, exist_ok=True)

print("PROJECT:", PROJECT)
print("DEVICE:", DEVICE)
print("MODELS exists:", MODELS.exists())


## 逐行語法解說

- **第 1 行 `from ... import`**：只匯入 pathlib 中的 Path 類別。
- **第 3 行 `Path`**：建立路徑物件，不會立刻建立資料夾。
- **第 4 行 `Path / str`**：Path 重新定義 /，用來串接路徑。
- **第 6 行 `A if 條件 else B`**：條件成立取 0，否則取 cpu。
- **第 7 行 `mkdir`**：parents 建上層；exist_ok 允許資料夾已存在。
- **第 11 行 `exists`**：回傳資料夾是否存在的 bool。


## 修改挑戰

- [ ] 將 PROJECT 改成 Google Drive 路徑。
- [ ] 把 exist_ok 改成 False 並重跑，閱讀錯誤訊息。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert MODELS.exists() and DEVICE == "cpu"


# 第 3 單元 字元類別與 data yaml

**學習目標**：用 list、enumerate、dict comprehension 與函式產生 YOLO 設定。

**語法焦點**：list、enumerate、字典生成式、def、return、檔案寫入

## 執行前預測

- [ ] CHAR_NAMES[10] 的結果為何？
- [ ] 函式回傳值的型態為何？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
from pathlib import Path
import yaml

CHARACTERS = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
CHAR_NAMES = {i: ch for i, ch in enumerate(CHARACTERS)}

def write_data_yaml(root, names):
    config = {
        "path": str(root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": names,
    }
    path = root / "data.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    return path

demo_root = Path("/content/demo_dataset")
demo_root.mkdir(parents=True, exist_ok=True)
yaml_path = write_data_yaml(demo_root, {0: "plate"})
print(yaml_path.read_text())


## 逐行語法解說

- **第 4 行 `list(str)`**：把可迭代字串拆成 36 個字元。
- **第 5 行 `dict comprehension`**：enumerate 同時給索引與內容。
- **第 7 行 `def`**：定義可重複呼叫的函式；此時尚未執行。
- **第 8–14 行 `dict literal`**：建立 YOLO 所需欄位。
- **第 16 行 `safe_dump`**：把 dict 序列化為 YAML 文字。
- **第 17 行 `return`**：把產生的 Path 交回呼叫端。
- **第 21 行 `函式呼叫`**：傳入資料集根目錄與類別字典。


## 修改挑戰

- [ ] 產生字元資料集 YAML。
- [ ] 故意改錯 train 路徑並說明後續會在哪裡失敗。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert len(CHAR_NAMES) == 36 and yaml_path.exists()


# 第 4 單元 標註稽核與控制流程

**學習目標**：練習雙層 for、if、continue、append、型別轉換與 any。

**語法焦點**：glob、for、enumerate、split、continue、list comprehension、any

## 執行前預測

- [ ] 第二筆資料會產生哪一種 issue？
- [ ] 第三筆會在哪個判斷被攔截？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
from pathlib import Path

def audit_rows(rows, class_count):
    issues = []
    for line_no, row in enumerate(rows, start=1):
        parts = row.split()
        if len(parts) != 5:
            issues.append((line_no, "欄位數不是 5"))
            continue
        try:
            class_id = int(parts[0])
            coords = [float(v) for v in parts[1:]]
        except ValueError as error:
            issues.append((line_no, f"數值格式錯誤: {error}"))
            continue
        if not 0 <= class_id < class_count:
            issues.append((line_no, "class 超界"))
        if any(v < 0 or v > 1 for v in coords):
            issues.append((line_no, "座標超界"))
    return issues

demo_rows = ["0 0.5 0.5 0.2 0.1", "0 1.2 0.5 0.2 0.1", "x 0.5 0.5"]
print(audit_rows(demo_rows, 1))


## 逐行語法解說

- **第 4 行 `issues=[]`**：建立容器，累積所有錯誤而非遇錯即停止。
- **第 5 行 `enumerate`**：同時取得人類可讀行號與內容。
- **第 6 行 `split`**：依空白把一行切成欄位。
- **第 7–9 行 `continue`**：欄位錯誤時跳到下一輪。
- **第 10–15 行 `try except`**：攔截 int/float 的 ValueError。
- **第 12 行 `list comprehension`**：把四個座標轉為 float。
- **第 16 行 `連鎖比較`**：等同 0 <= id and id < count。
- **第 18 行 `any`**：任一座標超界即成立。
- **第 20 行 `return`**：回傳完整問題清單。


## 修改挑戰

- [ ] 新增空白列檢查。
- [ ] 把 issue 改成 dict，加入 severity 欄位。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert len(audit_rows(demo_rows, 1)) == 2


# 第 5 單元 模型物件 方法與訓練參數

**學習目標**：理解 class、instance、method 與 keyword arguments。

**語法焦點**：類別、物件、方法呼叫、關鍵字參數、回傳值

## 執行前預測

- [ ] RUN_TRAINING=False 時哪些行不會執行？
- [ ] batch 減半對顯存有何影響？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ultralytics import YOLO
    plate_model = YOLO("yolo11n.pt")
    train_result = plate_model.train(
        data=str(PLATE_YAML),
        epochs=100,
        imgsz=960,
        batch=8,
        device=DEVICE,
        project=str(PROJECT / "runs"),
        name="plate_yolo11n",
        patience=20,
        seed=42,
    )
else:
    print("語法示範模式：將 RUN_TRAINING 改為 True 才會訓練")


## 逐行語法解說

- **第 1 行 `功能開關`**：避免沒有資料或 GPU 時誤啟動長時間訓練。
- **第 3 行 `if`**：只有 True 才執行縮排區塊。
- **第 5 行 `YOLO(...)`**：以類別建立 plate_model 物件。
- **第 6 行 `train`**：呼叫物件方法。
- **第 7–15 行 `keyword args`**：以參數名傳值，清楚且不依位置。
- **第 8 行 `epochs`**：完整走訪訓練資料的輪數。
- **第 9 行 `imgsz`**：輸入解析度，影響小物件與顯存。
- **第 10 行 `batch`**：每次更新使用的影像數。
- **第 16–17 行 `else`**：語法模式只顯示提示，不訓練。


## 修改挑戰

- [ ] 建立 char_model 訓練區塊，將 fliplr 設為 0。
- [ ] 若 CUDA OOM，依序修改 batch 與 imgsz。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert RUN_TRAINING is False


# 第 6 單元 字串 正規表示式與布林規則

**學習目標**：正規化模型輸出並建立候選車牌規則。

**語法焦點**：str.upper、re.sub、fullmatch、generator、any、and、assert

## 執行前預測

- [ ] NY-3509 正規化後為何？
- [ ] ABCDEFG 為何被拒絕？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
import re

def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", text.upper())

def valid_candidate(text):
    length_ok = bool(re.fullmatch(r"[A-Z0-9]{5,8}", text))
    has_digit = any(ch.isdigit() for ch in text)
    has_letter = any(ch.isalpha() for ch in text)
    return length_ok and has_digit and has_letter

samples = ["ab-c123", "1234", "ABCDEFG", "NY-3509"]
for sample in samples:
    text = normalize_plate(sample)
    print(sample, "->", text, valid_candidate(text))


## 逐行語法解說

- **第 3–4 行 `normalize_plate`**：先 upper，再用 re.sub 刪除非英數字元。
- **第 7 行 `fullmatch`**：整個字串必須是 5 到 8 個英數字。
- **第 8 行 `generator + any`**：至少一個字元 isdigit 為 True。
- **第 9 行 `isalpha`**：至少一個英文字母。
- **第 10 行 `and`**：三個條件都成立才接受。
- **第 12 行 `list`**：準備多種測試輸入。
- **第 13–15 行 `for`**：逐一正規化、驗證與印出。


## 修改挑戰

- [ ] 新增只允許 6 到 7 碼的規則。
- [ ] 不要無條件把 O 改成 0，說明原因。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert normalize_plate("ab-c123") == "ABC123" and valid_candidate("ABC123")


# 第 7 單元 字元排序 Lambda 與 List Comprehension

**學習目標**：依座標把偵測字元還原為正確順序。

**語法焦點**：sorted、key、lambda、slice、list comprehension、np.argmax

## 執行前預測

- [ ] demo 的輸出為何？
- [ ] sorted 是否修改原本 demo？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
import numpy as np

def order_characters(items, two_lines=False):
    if not items:
        return []
    if not two_lines or len(items) < 2:
        return sorted(items, key=lambda item: item[0])
    by_y = sorted(items, key=lambda item: item[1])
    gaps = [by_y[i + 1][1] - by_y[i][1]
            for i in range(len(by_y) - 1)]
    split = int(np.argmax(gaps)) + 1
    top = sorted(by_y[:split], key=lambda item: item[0])
    bottom = sorted(by_y[split:], key=lambda item: item[0])
    return top + bottom

demo = [(80, 20, 10, "C", .9), (20, 20, 10, "A", .9), (50, 20, 10, "B", .9)]
print("".join(item[3] for item in order_characters(demo)))


## 逐行語法解說

- **第 4 行 `not items`**：空 list 在布林情境為 False。
- **第 6–7 行 `sorted + lambda`**：以 item[0] 的 x 座標由左到右排序。
- **第 8 行 `item[1]`**：雙行時先依 y 座標由上到下排序。
- **第 9–10 行 `list comprehension`**：計算相鄰 y 差距。
- **第 11 行 `argmax`**：取得最大間距索引並加一成切割點。
- **第 12–13 行 `slice`**：切出上下兩排並各自依 x 排序。
- **第 14 行 `list + list`**：串接兩排的新清單。
- **第 17 行 `join`**：取 tuple 第四欄字元並合併。


## 修改挑戰

- [ ] 建立兩行測試資料。
- [ ] 比較 sorted(items) 與 items.sort() 的回傳值。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert "".join(x[3] for x in order_characters(demo)) == "ABC"


# 第 8 單元 雙階段推論與函式資料流

**學習目標**：把影像、車牌框、裁切、字元與紀錄串成主流程。

**語法焦點**：函式參數、copy、shape、zip、map、continue、array slicing、dict 更新

## 執行前預測

- [ ] result.boxes=None 時回傳什麼？
- [ ] 若 x1=-5，限制後是多少？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
def recognize_plates_bgr(image_bgr, plate_model, char_model):
    output = image_bgr.copy()
    height, width = image_bgr.shape[:2]
    result = plate_model.predict(image_bgr, conf=PLATE_CONF,
                                 device=DEVICE, verbose=False)[0]
    records = []
    if result.boxes is None:
        return output, records
    pairs = zip(result.boxes.xyxy.cpu().numpy(),
                result.boxes.conf.cpu().numpy())
    for box, score in pairs:
        x1, y1, x2, y2 = map(int, box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(width, x2), min(height, y2)
        if x2 - x1 < 12 or y2 - y1 < 8:
            continue
        crop = image_bgr[y1:y2, x1:x2]
        record = recognize_one_crop(crop, char_model)
        record.update(box=(x1,y1,x2,y2), plate_conf=float(score))
        records.append(record)
    return output, records


## 逐行語法解說

- **第 2 行 `copy`**：建立可繪圖副本，不直接改輸入陣列。
- **第 3 行 `shape[:2]`**：取得 height、width 並拆包。
- **第 4–5 行 `predict()[0]`**：取得第一張影像的 Results。
- **第 6–8 行 `early return`**：沒有框就提早回傳空 records。
- **第 9–10 行 `zip`**：同步配對每個框與信心。
- **第 12 行 `map(int)`**：四座標轉成整數並拆包。
- **第 13–14 行 `max/min`**：將框限制在影像邊界。
- **第 15–16 行 `continue`**：忽略太小框但繼續處理其他框。
- **第 17 行 `array slicing`**：NumPy 影像先 y 後 x 裁切。
- **第 18–20 行 `dict update`**：加入框與信心後 append。
- **第 21 行 `多回傳值`**：回傳 tuple，可由兩變數拆包。


## 修改挑戰

- [ ] 把 continue 改成 return，說明多車牌影像的錯誤。
- [ ] 在 record 新增 crop_shape。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
print("本單元需載入模型後，以實際影像驗收")


# 第 9 單元 字元辨識子函式與巢狀資料

**學習目標**：讀懂 YOLO boxes 的框、信心、類別配對與字串組合。

**語法焦點**：三序列 zip、tuple unpacking、dict lookup、join、min default

## 執行前預測

- [ ] items 空白時 text、weakest、accepted 各為何？
- [ ] 為何用最低字元信心而不是最高？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
def recognize_one_crop(crop, char_model):
    result = char_model.predict(crop, conf=CHAR_CONF,
                                device=DEVICE, verbose=False)[0]
    items = []
    if result.boxes is not None:
        arrays = zip(result.boxes.xyxy.cpu().numpy(),
                     result.boxes.conf.cpu().numpy(),
                     result.boxes.cls.cpu().numpy())
        for xyxy, confidence, class_id in arrays:
            x1, y1, x2, y2 = xyxy
            label = str(result.names[int(class_id)]).upper()
            if label == "-":
                continue
            items.append(((x1+x2)/2, (y1+y2)/2,
                          y2-y1, label, float(confidence)))
    ordered = order_characters(items)
    text = normalize_plate("".join(item[3] for item in ordered))
    weakest = min((item[4] for item in ordered), default=0.0)
    accepted = valid_candidate(text) and weakest >= TEXT_CONF
    return {"plate": text, "character_min_conf": weakest,
            "accepted": accepted}


## 逐行語法解說

- **第 5 行 `is not None`**：確認模型確實有 boxes。
- **第 6–8 行 `三序列 zip`**：同步配對框、信心與類別。
- **第 9 行 `tuple unpack`**：每輪拆成 xyxy、confidence、class_id。
- **第 11 行 `names lookup`**：用類別索引取得字元名稱。
- **第 12–13 行 `continue`**：忽略連字號類別。
- **第 14–15 行 `tuple`**：保存中心、字高、標籤與信心。
- **第 17 行 `join generator`**：依排序結果合併字元。
- **第 18 行 `min default`**：空集合時回傳 0.0，避免 ValueError。
- **第 19 行 `and`**：格式與最低字元信心都通過。
- **第 20–21 行 `dict return`**：以具名欄位回傳結果。


## 修改挑戰

- [ ] 新增 character_count 欄位。
- [ ] 保留 raw_text 與 normalized text 兩個欄位。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert min([], default=0.0) == 0.0


# 第 10 單元 圖片上傳與批次防呆

**學習目標**：以 for 逐檔處理 bytes，解碼影像並跳過壞檔。

**語法焦點**：dict.items、np.frombuffer、is None、continue、多回傳值

## 執行前預測

- [ ] uploaded 的型態為何？
- [ ] 壞檔之後其他檔案是否仍會處理？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
# Colab 執行區塊
from google.colab import files
import cv2, numpy as np

uploaded = files.upload()
for filename, payload in uploaded.items():
    data = np.frombuffer(payload, dtype=np.uint8)
    image = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if image is None:
        print(f"無法解碼: {filename}")
        continue
    annotated, records = recognize_plates_bgr(
        image, plate_model, char_model)
    print(filename, records)


## 逐行語法解說

- **第 5 行 `files.upload`**：回傳 filename 到 bytes 的 dict。
- **第 6 行 `items`**：每輪同時取得檔名與內容。
- **第 7 行 `frombuffer`**：把 bytes 視為 uint8 陣列，尚未解碼。
- **第 8 行 `imdecode`**：轉成 OpenCV BGR 影像。
- **第 9–11 行 `防呆`**：壞檔只跳過本輪，不中止整批。
- **第 12–13 行 `多行呼叫`**：括號內可自然換行。
- **第 14 行 `records`**：印出每張圖的辨識紀錄。


## 修改挑戰

- [ ] 限制只接受 jpg、jpeg、png。
- [ ] 為每張圖記錄處理時間。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
print("本單元在 Colab 上傳圖片後驗收")


# 第 11 單元 影片 while 迴圈與資源釋放

**學習目標**：逐幀讀取影片，依取樣頻率推論，並確保釋放資源。

**語法焦點**：while True、break、+=、%、try finally

## 執行前預測

- [ ] SAMPLE_EVERY=3 時哪些影格推論？
- [ ] break 與 return 有何不同？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
cap = cv2.VideoCapture(str(INPUT_VIDEO))
writer = make_writer(cap, OUTPUT_VIDEO)
frame_index = 0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_index += 1
        if frame_index % SAMPLE_EVERY == 0:
            frame, records = recognize_plates_bgr(
                frame, plate_model, char_model)
        writer.write(frame)
finally:
    cap.release()
    writer.release()


## 逐行語法解說

- **第 1 行 `VideoCapture`**：建立影片讀取物件。
- **第 3 行 `counter`**：追蹤目前影格編號。
- **第 5 行 `try finally`**：無論成功或錯誤都執行 finally。
- **第 6 行 `while True`**：持續讀取，靠 break 結束。
- **第 7 行 `雙回傳`**：ok 表示成功，frame 是影像。
- **第 8–9 行 `break`**：讀完影片或失敗時離開迴圈。
- **第 10 行 `+=`**：等同 frame_index = frame_index + 1。
- **第 11 行 `%`**：餘數為 0 的影格才推論。
- **第 15 行 `write`**：所有影格都寫入輸出。
- **第 16–18 行 `release`**：避免輸出檔損壞或資源未釋放。


## 修改挑戰

- [ ] 加入 cap.isOpened 檢查。
- [ ] 說明跳幀時為何需要追蹤器維持框。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert 6 % 3 == 0 and 5 % 3 != 0


# 第 12 單元 UI 狀態 事件與去重

**學習目標**：用巢狀 dict/list 管理 UI 欄位、授權結果與事件歷史。

**語法焦點**：mutable dict、in、append、set comprehension、len

## 執行前預測

- [ ] 相同車牌註冊兩次，unique_count 為多少？
- [ ] plate 不在 authorized 時 event 如何？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
state = {
    "gate_open": False,
    "current_plate": None,
    "unique_count": 0,
    "events": [],
}

def register_detection(state, record, authorized):
    plate = record["plate"]
    event = {
        "plate": plate,
        "authorized": plate in authorized,
        "confidence": record["character_min_conf"],
    }
    state["current_plate"] = plate
    state["events"].append(event)
    state["unique_count"] = len({e["plate"] for e in state["events"]})
    return event

authorized = {"ABC1234"}
demo_record = {"plate":"ABC1234", "character_min_conf":.91}
print(register_detection(state, demo_record, authorized))
print(state)


## 逐行語法解說

- **第 1–6 行 `state dict`**：集中保存 UI 狀態。
- **第 8 行 `參數 state`**：明確傳入可修改物件，便於測試。
- **第 11 行 `in`**：檢查車牌是否存在授權 set。
- **第 14 行 `dict 更新`**：改變目前畫面車牌。
- **第 15 行 `nested append`**：取出 events list 後加入 event。
- **第 16 行 `set comprehension`**：去除重複車牌再計數。
- **第 17 行 `return`**：回傳本次事件供 UI 或通知使用。


## 修改挑戰

- [ ] 新增 timestamp 與 camera_id。
- [ ] 加入連續三次相同車牌才開閘的計數。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert state["unique_count"] == 1 and state["events"][0]["authorized"]


# 第 13 單元 Exact Match 與 CER

**學習目標**：用 zip、generator、sum 與動態規劃評估整牌文字。

**語法焦點**：range、enumerate、bool 算術、min、zip、map、max 防除零

## 執行前預測

- [ ] ABC123 與 ABC128 的 edit distance 為何？
- [ ] 兩筆中一筆完全相同，Exact Match 為何？

先把答案寫在下方文字格，再執行程式。


### 我的預測

- 關鍵變數與型態：
- 預期輸出：
- 我不確定的語法：


## 可執行程式


In [ ]:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current = [i]
        for j, char_b in enumerate(b, start=1):
            insert_cost = current[-1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j-1] + (char_a != char_b)
            current.append(min(insert_cost, delete_cost, replace_cost))
        previous = current
    return previous[-1]

def evaluate_texts(truths, predictions):
    if len(truths) != len(predictions):
        raise ValueError("truths 與 predictions 數量不同")
    pairs = list(zip(truths, predictions))
    exact = sum(t == p for t, p in pairs) / max(1, len(pairs))
    errors = sum(edit_distance(t, p) for t, p in pairs)
    chars = sum(map(len, truths))
    cer = errors / max(1, chars)
    return {"exact_match": exact, "cer": cer}

print(evaluate_texts(["ABC123", "NY3509"], ["ABC128", "NY3509"]))


## 逐行語法解說

- **第 2 行 `初始列`**：建立 0 到 len(b) 的距離。
- **第 3–5 行 `巢狀迴圈`**：逐一比較兩字串的字元。
- **第 6–8 行 `三種成本`**：插入、刪除、替換。
- **第 8 行 `bool 算術`**：相同 False=0，不同 True=1。
- **第 9 行 `min`**：保留最低累積成本。
- **第 14–15 行 `raise`**：輸入筆數不同時主動報錯。
- **第 16 行 `zip`**：將真值與預測一一配對。
- **第 17 行 `sum(bool)`**：True 以 1 累加完全相同筆數。
- **第 18 行 `generator sum`**：加總每對編輯距離。
- **第 19 行 `map(len)`**：加總真值字元數。
- **第 20 行 `max(1,...)`**：避免除以零。


## 修改挑戰

- [ ] 新增每筆 CER 明細。
- [ ] 以白天、夜間分組計算指標。

每次只修改一個條件或參數，保留修改前後輸出。


### 修改紀錄

- 修改項目：
- 修改前輸出：
- 修改後輸出：
- 我對差異的解釋：


## 驗收區塊


In [ ]:
assert edit_distance("ABC123", "ABC128") == 1


# 綜合專題驗收

- [ ] 能逐行解釋 `recognize_plates_bgr`
- [ ] 兩個模型皆可載入
- [ ] 圖片輸出 `records`
- [ ] 影片逐幀處理並正確釋放資源
- [ ] UI 狀態可記錄目前車牌、授權結果與去重數
- [ ] 至少 8 個 `assert`
- [ ] 完成 Exact Match 與 CER
- [ ] 整理成功與失敗案例

## 專題反思

1. 本專案最重要的 Python 資料型態是什麼？為什麼？
2. 哪個錯誤來自模型，哪個錯誤來自排序或規則？
3. 下一輪要優先修改資料、模型、程式或門檻？證據是什麼？
